<a href="https://colab.research.google.com/github/AnishSharma1/Science-Fair-Project/blob/main/AlphaFold2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="200" align="right" style="height:240px">

##ColabFold v1.6.2: AlphaFold2 using MMseqs2

Easy to use protein structure and complex prediction using [AlphaFold2](https://www.nature.com/articles/s41586-021-03819-2) and [Alphafold2-multimer](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1). Sequence alignments/templates are generated through [MMseqs2](mmseqs.com) and [HHsearch](https://github.com/soedinglab/hh-suite). For more details, see <a href="#Instructions">bottom</a> of the notebook, checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold) and [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5).

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/AlphaFold2.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/AlphaFold2.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/AlphaFold2.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/AlphaFold2.ipynb)

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

In [10]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
from google.colab import files
import os
import re
import hashlib
import random
from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"
def add_hash(x,y):
  return x+"_"+hashlib.sha1(y.encode()).hexdigest()[:5]
query_sequence = 'EHVIIQAEFYLNPDQSGEFMFDFDGDEIFHVDMAKKETVWRLEEFGRFASFEAQGALANIAVDKANLEIMTKRSNYTPITNVPPEVTVLTNSPVELREPNVLICFIDKFTPPVVNVTWLRNGKPVTTGVSETVFLPREDHLFRKFHYLPFLPSTEDVYDCRVEHWGLDEPLLKHWEFD:DTRPRFLWQPKRECHFFNGTERVRFLDRYFYNQEESVRFDSDVGEFRAVTELGRPDAEYWNSQKDILEQARAAVDTYCRHNYGVGESFTVQRRVQPKVTVYPSKTQPLQHHNLLVCSVSGFYPGSIEVRWFLNGQEEKAGMVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSVTSPLTVEWRA:IMNILRIYYSPSIM' #@param {type:"string"}
#@markdown  - Use `:` to specify inter-protein chainbreaks for **modeling complexes** (supports homo- and hetro-oligomers). For example **PI...SK:PI...SK** for a homodimer
jobname = 'EBV_TCELL_2268741_pMHCII' #@param {type:"string"}
# number of models to use
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
#@markdown - specify how many of the top ranked structures to relax using amber
template_mode = "none" #@param ["none", "pdb100","custom"]
#@markdown - `none` = no template information is used. `pdb100` = detect templates in pdb100 (see [notes](#pdb100)). `custom` - upload and search own templates (PDB or mmCIF format, see [notes](#custom_templates))
use_amber = num_relax > 0
# remove whitespaces
query_sequence = "".join(query_sequence.split())
basejobname = "".join(jobname.split())
basejobname = re.sub(r'\W+', '', basejobname)
jobname = add_hash(basejobname, query_sequence)
# check if directory with jobname exists
def check(folder):
  if os.path.exists(folder):
    return False
  else:
    return True
if not check(jobname):
  n = 0
  while not check(f"{jobname}_{n}"): n += 1
  jobname = f"{jobname}_{n}"
# make directory to save results
os.makedirs(jobname, exist_ok=True)
# save queries
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
  text_file.write(f"id,sequence\n{jobname},{query_sequence}")
if template_mode == "pdb100":
  use_templates = True
  custom_template_path = None
elif template_mode == "custom":
  custom_template_path = os.path.join(jobname,f"template")
  os.makedirs(custom_template_path, exist_ok=True)
  uploaded = files.upload()
  use_templates = True
  for fn in uploaded.keys():
    os.rename(fn,os.path.join(custom_template_path,fn))
else:
  custom_template_path = None
  use_templates = False
print("jobname",jobname)
print("sequence",query_sequence)
print("length",len(query_sequence.replace(":","")))

jobname EBV_TCELL_2268741_pMHCII_202a0_5
sequence EHVIIQAEFYLNPDQSGEFMFDFDGDEIFHVDMAKKETVWRLEEFGRFASFEAQGALANIAVDKANLEIMTKRSNYTPITNVPPEVTVLTNSPVELREPNVLICFIDKFTPPVVNVTWLRNGKPVTTGVSETVFLPREDHLFRKFHYLPFLPSTEDVYDCRVEHWGLDEPLLKHWEFD:DTRPRFLWQPKRECHFFNGTERVRFLDRYFYNQEESVRFDSDVGEFRAVTELGRPDAEYWNSQKDILEQARAAVDTYCRHNYGVGESFTVQRRVQPKVTVYPSKTQPLQHHNLLVCSVSGFYPGSIEVRWFLNGQEEKAGMVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSVTSPLTVEWRA:IMNILRIYYSPSIM
length 381


In [12]:
# Upload one prepared batch file:
# - processed/pmhc_colabfold_inputs.fasta
# - processed/pmhc_colabfold_batch.csv
from google.colab import files, drive
import csv
import hashlib
import json
import os
import shutil
from pathlib import Path

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError(f"Upload exactly one batch file; found {list(uploaded)}")
uploaded_file = Path(next(iter(uploaded)))


def parse_fasta(path: Path) -> list[dict[str, str]]:
    rows = []
    header = None
    seq_lines = []
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if header is not None:
                rows.append({"id": header, "sequence": "".join(seq_lines)})
            header = line[1:].split("|", 1)[0].strip()
            seq_lines = []
        else:
            seq_lines.append(line)
    if header is not None:
        rows.append({"id": header, "sequence": "".join(seq_lines)})
    return rows


def parse_batch(path: Path) -> list[dict[str, str]]:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        with path.open(newline="") as handle:
            rows = list(csv.DictReader(handle))
        if not rows or set(rows[0]) != {"id", "sequence"}:
            raise ValueError("The CSV must have exactly two columns: id,sequence")
        return rows
    if suffix in {".fasta", ".fa", ".faa"}:
        rows = parse_fasta(path)
        if not rows:
            raise ValueError("The FASTA file did not contain any sequences")
        return rows
    raise ValueError("Upload a .csv, .fasta, .fa, or .faa batch file")


rows = parse_batch(uploaded_file)
if len({row["id"] for row in rows}) != len(rows):
    raise ValueError("Candidate IDs must be unique")
for row in rows:
    row["id"] = row["id"].strip()
    row["sequence"] = "".join(row["sequence"].split()).upper()
    if not row["id"]:
        raise ValueError("Every candidate must have a non-empty ID")
    if row["sequence"].count(":") != 2:
        raise ValueError(f"{row['id']} does not contain DRA:DRB:peptide chains")
    if any(aa not in "ACDEFGHIKLMNPQRSTVWY:" for aa in row["sequence"]):
        raise ValueError(f"{row['id']} contains a non-standard amino-acid symbol")

BATCH_NAME = "ebv_ms_pmhc_batch"
RUN_DIR = Path("/content") / BATCH_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
batch_csv = RUN_DIR / "pmhc_colabfold_batch.csv"
shutil.copy2(uploaded_file, RUN_DIR / uploaded_file.name)
with batch_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["id", "sequence"])
    writer.writeheader()
    writer.writerows(rows)

settings = {
    "model_type": "alphafold2_multimer_v3",
    "msa_mode": "mmseqs2_uniref_env",
    "pair_mode": "unpaired_paired",
    "num_models": 5,
    "num_recycles": 3,
    "num_seeds": 1,
    "num_relax": 0,
    "save_all": True,
    "calc_extra_ptm": True,
}
(RUN_DIR / "run_metadata.json").write_text(
    json.dumps(
        {
            "candidate_count": len(rows),
            "candidate_ids": [row["id"] for row in rows],
            "input_sha256": hashlib.sha256(batch_csv.read_bytes()).hexdigest(),
            "settings": settings,
        },
        indent=2,
    )
)

# These imports are supplied by the ColabFold installation cell.
try:
    from colabfold.batch import get_queries, run, set_model_type
    from colabfold.download import download_alphafold_params
    from colabfold.utils import setup_logging
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "ColabFold is not installed in this active runtime. Run the notebook's "
        "'Install dependencies' cell first, then rerun this batch cell."
    ) from exc

queries, is_complex = get_queries(str(batch_csv))
if not is_complex:
    raise ValueError("The uploaded batch was not recognized as a complex")
model_type = set_model_type(is_complex, settings["model_type"])
setup_logging(RUN_DIR / "log.txt")
download_alphafold_params(model_type, Path("/content"))

run(
    queries=queries,
    result_dir=str(RUN_DIR),
    use_templates=False,
    custom_template_path=None,
    num_relax=settings["num_relax"],
    msa_mode=settings["msa_mode"],
    model_type=model_type,
    num_models=settings["num_models"],
    num_recycles=settings["num_recycles"],
    relax_max_iterations=200,
    recycle_early_stop_tolerance=0.5,
    num_seeds=settings["num_seeds"],
    use_dropout=False,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=Path("/content"),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=settings["pair_mode"],
    pairing_strategy="greedy",
    stop_at_score=100.0,
    prediction_callback=None,
    dpi=200,
    zip_results=False,
    save_all=settings["save_all"],
    max_msa=None,
    use_cluster_profile=False,
    input_features_callback=None,
    save_recycles=False,
    user_agent="colabfold/google-colab-batch",
    calc_extra_ptm=settings["calc_extra_ptm"],
)

# Preserve the full run in Drive for later QA and publication analysis.
drive.mount("/content/drive")
drive_dir = Path("/content/drive/MyDrive") / BATCH_NAME
if drive_dir.exists():
    from datetime import datetime
    drive_dir = drive_dir.with_name(
        f"{BATCH_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    )
shutil.copytree(RUN_DIR, drive_dir)
archive = shutil.make_archive(str(drive_dir), "zip", root_dir=drive_dir.parent, base_dir=drive_dir.name)
print(f"Completed {len(rows)} candidates")
print(f"Drive folder: {drive_dir}")
print(f"Drive archive: {archive}")


Saving pmhc_colabfold_inputs.fasta to pmhc_colabfold_inputs (4).fasta


ModuleNotFoundError: ColabFold is not installed in this active runtime. Run the notebook's 'Install dependencies' cell first, then rerun this batch cell.

In [ ]:
#@title Install dependencies
%%time
import os
USE_AMBER = use_amber
USE_TEMPLATES = use_templates
PYTHON_VERSION = python_version

if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  # hack to fix TF crash
  os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so /usr/local/lib/python3.*/dist-packages/tensorflow/lite/python/*/*.so")
  os.system("touch COLABFOLD_READY")

if USE_AMBER or USE_TEMPLATES:
  if not os.path.isfile("CONDA_READY"):
    print("installing conda...")
    os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
    os.system("mamba config --set auto_update_conda false")
    os.system("touch CONDA_READY")

if USE_TEMPLATES and not os.path.isfile("HH_READY") and USE_AMBER and not os.path.isfile("AMBER_READY"):
  print("installing hhsuite and amber...")
  os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
  os.system("touch HH_READY")
  os.system("touch AMBER_READY")
else:
  if USE_TEMPLATES and not os.path.isfile("HH_READY"):
    print("installing hhsuite...")
    os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python='{PYTHON_VERSION}'")
    os.system("touch HH_READY")
  if USE_AMBER and not os.path.isfile("AMBER_READY"):
    print("installing amber...")
    os.system(f"mamba install -y -c conda-forge openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
    os.system("touch AMBER_READY")

In [ ]:
#@markdown ### MSA options (custom MSA upload, single sequence, pairing mode)
msa_mode = "mmseqs2_uniref_env" #@param ["mmseqs2_uniref_env", "mmseqs2_uniref","single_sequence","custom"]
pair_mode = "unpaired_paired" #@param ["unpaired_paired","paired","unpaired"] {type:"string"}
#@markdown - "unpaired_paired" = pair sequences from same species + unpaired MSA, "unpaired" = seperate MSA for each chain, "paired" - only use paired sequences.

# decide which a3m to use
if "mmseqs2" in msa_mode:
  a3m_file = os.path.join(jobname,f"{jobname}.a3m")

elif msa_mode == "custom":
  a3m_file = os.path.join(jobname,f"{jobname}.custom.a3m")
  if not os.path.isfile(a3m_file):
    custom_msa_dict = files.upload()
    custom_msa = list(custom_msa_dict.keys())[0]
    header = 0
    import fileinput
    for line in fileinput.FileInput(custom_msa,inplace=1):
      if line.startswith(">"):
         header = header + 1
      if not line.rstrip():
        continue
      if line.startswith(">") == False and header == 1:
         query_sequence = line.rstrip()
      print(line, end='')

    os.rename(custom_msa, a3m_file)
    queries_path=a3m_file
    print(f"moving {custom_msa} to {a3m_file}")

else:
  a3m_file = os.path.join(jobname,f"{jobname}.single_sequence.a3m")
  with open(a3m_file, "w") as text_file:
    text_file.write(">1\n%s" % query_sequence)

In [ ]:
#@markdown ### Advanced settings
model_type = "auto" #@param ["auto", "alphafold2_ptm", "alphafold2_multimer_v1", "alphafold2_multimer_v2", "alphafold2_multimer_v3", "deepfold_v1", "alphafold2"]
#@markdown - if `auto` selected, will use `alphafold2_ptm` for monomer prediction and `alphafold2_multimer_v3` for complex prediction.
#@markdown Any of the mode_types can be used (regardless if input is monomer or complex).
num_recycles = "3" #@param ["auto", "0", "1", "3", "6", "12", "24", "48"]
#@markdown - if `auto` selected, will use `num_recycles=20` if `model_type=alphafold2_multimer_v3`, else `num_recycles=3` .
recycle_early_stop_tolerance = "auto" #@param ["auto", "0.0", "0.5", "1.0"]
#@markdown - if `auto` selected, will use `tol=0.5` if `model_type=alphafold2_multimer_v3` else `tol=0.0`.
relax_max_iterations = 200 #@param [0, 200, 2000] {type:"raw"}
#@markdown - max amber relax iterations, `0` = unlimited (AlphaFold2 default, can take very long)
pairing_strategy = "greedy" #@param ["greedy", "complete"] {type:"string"}
#@markdown - `greedy` = pair any taxonomically matching subsets, `complete` = all sequences have to match in one line.
calc_extra_ptm = False #@param {type:"boolean"}
#@markdown - return pairwise chain iptm/actifptm

#@markdown #### Sample settings
#@markdown -  enable dropouts and increase number of seeds to sample predictions from uncertainty of the model.
#@markdown -  decrease `max_msa` to increase uncertainity
max_msa = "auto" #@param ["auto", "512:1024", "256:512", "64:128", "32:64", "16:32"]
num_seeds = 1 #@param [1,2,4,8,16] {type:"raw"}
use_dropout = False #@param {type:"boolean"}

num_recycles = None if num_recycles == "auto" else int(num_recycles)
recycle_early_stop_tolerance = None if recycle_early_stop_tolerance == "auto" else float(recycle_early_stop_tolerance)
if max_msa == "auto": max_msa = None

#@markdown #### Save settings
save_all = False #@param {type:"boolean"}
save_recycles = False #@param {type:"boolean"}
save_to_google_drive = False #@param {type:"boolean"}
#@markdown -  if the save_to_google_drive option was selected, the result zip will be uploaded to your Google Drive
dpi = 200 #@param {type:"integer"}
#@markdown - set dpi for image resolution

if save_to_google_drive:
  from pydrive2.drive import GoogleDrive
  from pydrive2.auth import GoogleAuth
  from google.colab import auth
  from oauth2client.client import GoogleCredentials
  auth.authenticate_user()
  gauth = GoogleAuth()
  gauth.credentials = GoogleCredentials.get_application_default()
  drive = GoogleDrive(gauth)
  print("You are logged into Google Drive and are good to go!")

#@markdown Don't forget to hit `Runtime` -> `Run all` after updating the form.

In [ ]:
#@title Run Prediction
display_images = True #@param {type:"boolean"}

import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from Bio import BiopythonDeprecationWarning
warnings.simplefilter(action='ignore', category=BiopythonDeprecationWarning)
from pathlib import Path
from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type
from colabfold.plot import plot_msa_v2

import os
import numpy as np
try:
  K80_chk = os.popen('nvidia-smi | grep "Tesla K80" | wc -l').read()
except:
  K80_chk = "0"
  pass
if "1" in K80_chk:
  print("WARNING: found GPU Tesla K80: limited to total length < 1000")
  if "TF_FORCE_UNIFIED_MEMORY" in os.environ:
    del os.environ["TF_FORCE_UNIFIED_MEMORY"]
  if "XLA_PYTHON_CLIENT_MEM_FRACTION" in os.environ:
    del os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]

from colabfold.colabfold import plot_protein
from pathlib import Path
import matplotlib.pyplot as plt

# For some reason we need that to get pdbfixer to import
if use_amber and f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

def input_features_callback(input_features):
  if display_images:
    plot_msa_v2(input_features)
    plt.show()
    plt.close()

def prediction_callback(protein_obj, length,
                        prediction_result, input_features, mode):
  model_name, relaxed = mode
  if not relaxed:
    if display_images:
      fig = plot_protein(protein_obj, Ls=length, dpi=150)
      plt.show()
      plt.close()

result_dir = jobname
log_filename = os.path.join(jobname,"log.txt")
setup_logging(Path(log_filename))

queries, is_complex = get_queries(queries_path)
model_type = set_model_type(is_complex, model_type)

if "multimer" in model_type and max_msa is not None:
  use_cluster_profile = False
else:
  use_cluster_profile = True

download_alphafold_params(model_type, Path("."))
results = run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    custom_template_path=custom_template_path,
    num_relax=num_relax,
    msa_mode=msa_mode,
    model_type=model_type,
    num_models=5,
    num_recycles=num_recycles,
    relax_max_iterations=relax_max_iterations,
    recycle_early_stop_tolerance=recycle_early_stop_tolerance,
    num_seeds=num_seeds,
    use_dropout=use_dropout,
    model_order=[1,2,3,4,5],
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=pair_mode,
    pairing_strategy=pairing_strategy,
    stop_at_score=float(100),
    prediction_callback=prediction_callback,
    dpi=dpi,
    zip_results=False,
    save_all=save_all,
    max_msa=max_msa,
    use_cluster_profile=use_cluster_profile,
    input_features_callback=input_features_callback,
    save_recycles=save_recycles,
    user_agent="colabfold/google-colab-main",
    calc_extra_ptm=calc_extra_ptm,
)
results_zip = f"{jobname}.result.zip"
os.system(f"zip -r {results_zip} {jobname}")

In [ ]:
#@title Display 3D structure {run: "auto"}
import py3Dmol
import glob
import matplotlib.pyplot as plt
from colabfold.colabfold import plot_plddt_legend
from colabfold.colabfold import pymol_color_list, alphabet_list
rank_num = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}
color = "lDDT" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

tag = results["rank"][0][rank_num - 1]
jobname_prefix = ".custom" if msa_mode == "custom" else ""
pdb_filename = f"{jobname}/{jobname}{jobname_prefix}_unrelaxed_{tag}.pdb"
pdb_file = glob.glob(pdb_filename)

def show_pdb(rank_num=1, show_sidechains=False, show_mainchains=False, color="lDDT"):
  model_name = f"rank_{rank_num}"
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',)
  view.addModel(open(pdb_file[0],'r').read(),'pdb')

  if color == "lDDT":
    view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
  elif color == "rainbow":
    view.setStyle({'cartoon': {'color':'spectrum'}})
  elif color == "chain":
    chains = len(queries[0][1]) + 1 if is_complex else 1
    for n,chain,color in zip(range(chains),alphabet_list,pymol_color_list):
       view.setStyle({'chain':chain},{'cartoon': {'color':color}})

  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                        {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                        {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})

  view.zoomTo()
  return view

show_pdb(rank_num, show_sidechains, show_mainchains, color).show()
if color == "lDDT":
  plot_plddt_legend().show()

In [ ]:
#@title Plots {run: "auto"}
from IPython.display import display, HTML
import base64
from html import escape

# see: https://stackoverflow.com/a/53688522
def image_to_data_url(filename):
  ext = filename.split('.')[-1]
  prefix = f'data:image/{ext};base64,'
  with open(filename, 'rb') as f:
    img = f.read()
  return prefix + base64.b64encode(img).decode('utf-8')

pae = ""
pae_file = os.path.join(jobname,f"{jobname}{jobname_prefix}_pae.png")
if os.path.isfile(pae_file):
    pae = image_to_data_url(pae_file)
cov = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_coverage.png"))
plddt = image_to_data_url(os.path.join(jobname,f"{jobname}{jobname_prefix}_plddt.png"))
display(HTML(f"""
<style>
  img {{
    float:left;
  }}
  .full {{
    max-width:100%;
  }}
  .half {{
    max-width:50%;
  }}
  @media (max-width:640px) {{
    .half {{
      max-width:100%;
    }}
  }}
</style>
<div style="max-width:90%; padding:2em;">
  <h1>Plots for {escape(jobname)}</h1>
  { '<!--' if pae == '' else '' }<img src="{pae}" class="full" />{ '-->' if pae == '' else '' }
  <img src="{cov}" class="half" />
  <img src="{plddt}" class="half" />
</div>
"""))

In [ ]:
#@title Package and download results
#@markdown If you are having issues downloading the result archive, try disabling your adblocker and run this cell again. If that fails click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

if msa_mode == "custom":
  print("Don't forget to cite your custom MSA generation method.")

files.download(f"{jobname}.result.zip")

if save_to_google_drive == True and drive:
  uploaded = drive.CreateFile({'title': f"{jobname}.result.zip"})
  uploaded.SetContentFile(f"{jobname}.result.zip")
  uploaded.Upload()
  print(f"Uploaded {jobname}.result.zip to Google Drive with ID {uploaded.get('id')}")

In [1]:
"""Paste this entire cell into the ColabFold notebook after installation.

It uploads either one multi-entry FASTA or one ``id,sequence`` CSV, runs all
pMHC-II candidates in one ColabFold-Multimer workflow, and copies the results
to Google Drive.
"""

# Upload one prepared batch file:
# - processed/pmhc_colabfold_inputs.fasta
# - processed/pmhc_colabfold_batch.csv
from google.colab import files, drive
import csv
import glob
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError(f"Upload exactly one batch file; found {list(uploaded)}")
uploaded_file = Path(next(iter(uploaded)))


def parse_fasta(path: Path) -> list[dict[str, str]]:
    rows = []
    header = None
    seq_lines = []
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if header is not None:
                rows.append({"id": header, "sequence": "".join(seq_lines)})
            header = line[1:].split("|", 1)[0].strip()
            seq_lines = []
        else:
            seq_lines.append(line)
    if header is not None:
        rows.append({"id": header, "sequence": "".join(seq_lines)})
    return rows


def parse_batch(path: Path) -> list[dict[str, str]]:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        with path.open(newline="") as handle:
            rows = list(csv.DictReader(handle))
        if not rows or set(rows[0]) != {"id", "sequence"}:
            raise ValueError("The CSV must have exactly two columns: id,sequence")
        return rows
    if suffix in {".fasta", ".fa", ".faa"}:
        rows = parse_fasta(path)
        if not rows:
            raise ValueError("The FASTA file did not contain any sequences")
        return rows
    raise ValueError("Upload a .csv, .fasta, .fa, or .faa batch file")


rows = parse_batch(uploaded_file)
if len({row["id"] for row in rows}) != len(rows):
    raise ValueError("Candidate IDs must be unique")
for row in rows:
    row["id"] = row["id"].strip()
    row["sequence"] = "".join(row["sequence"].split()).upper()
    if not row["id"]:
        raise ValueError("Every candidate must have a non-empty ID")
    if row["sequence"].count(":") != 2:
        raise ValueError(f"{row['id']} does not contain DRA:DRB:peptide chains")
    if any(aa not in "ACDEFGHIKLMNPQRSTVWY:" for aa in row["sequence"]):
        raise ValueError(f"{row['id']} contains a non-standard amino-acid symbol")

BATCH_NAME = "ebv_ms_pmhc_batch"
RUN_DIR = Path("/content") / BATCH_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
batch_csv = RUN_DIR / "pmhc_colabfold_batch.csv"
shutil.copy2(uploaded_file, RUN_DIR / uploaded_file.name)
with batch_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["id", "sequence"])
    writer.writeheader()
    writer.writerows(rows)

settings = {
    "model_type": "alphafold2_multimer_v3",
    "msa_mode": "mmseqs2_uniref_env",
    "pair_mode": "unpaired_paired",
    "num_models": 5,
    "num_recycles": 3,
    "num_seeds": 1,
    "num_relax": 0,
    "save_all": True,
    "calc_extra_ptm": True,
}
(RUN_DIR / "run_metadata.json").write_text(
    json.dumps(
        {
            "candidate_count": len(rows),
            "candidate_ids": [row["id"] for row in rows],
            "input_sha256": hashlib.sha256(batch_csv.read_bytes()).hexdigest(),
            "settings": settings,
        },
        indent=2,
    )
)

def ensure_colabfold():
    try:
        from colabfold.batch import get_queries, run, set_model_type
        from colabfold.download import download_alphafold_params
        from colabfold.utils import setup_logging
        return get_queries, run, set_model_type, download_alphafold_params, setup_logging
    except ModuleNotFoundError:
        print("ColabFold is missing in this runtime. Installing it now...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-warn-conflicts",
                "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold",
            ],
            check=True,
        )
        for pattern in [
            "/usr/local/lib/python*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so",
            "/usr/local/lib/python*/dist-packages/tensorflow/lite/python/*/*.so",
        ]:
            for match in glob.glob(pattern):
                try:
                    os.remove(match)
                except OSError:
                    pass
        from colabfold.batch import get_queries, run, set_model_type
        from colabfold.download import download_alphafold_params
        from colabfold.utils import setup_logging
        return get_queries, run, set_model_type, download_alphafold_params, setup_logging


get_queries, run, set_model_type, download_alphafold_params, setup_logging = ensure_colabfold()

queries, is_complex = get_queries(str(batch_csv))
if not is_complex:
    raise ValueError("The uploaded batch was not recognized as a complex")
model_type = set_model_type(is_complex, settings["model_type"])
setup_logging(RUN_DIR / "log.txt")
download_alphafold_params(model_type, Path("/content"))

run(
    queries=queries,
    result_dir=str(RUN_DIR),
    use_templates=False,
    custom_template_path=None,
    num_relax=settings["num_relax"],
    msa_mode=settings["msa_mode"],
    model_type=model_type,
    num_models=settings["num_models"],
    num_recycles=settings["num_recycles"],
    relax_max_iterations=200,
    recycle_early_stop_tolerance=0.5,
    num_seeds=settings["num_seeds"],
    use_dropout=False,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=Path("/content"),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=settings["pair_mode"],
    pairing_strategy="greedy",
    stop_at_score=100.0,
    prediction_callback=None,
    dpi=200,
    zip_results=False,
    save_all=settings["save_all"],
    max_msa=None,
    use_cluster_profile=False,
    input_features_callback=None,
    save_recycles=False,
    user_agent="colabfold/google-colab-batch",
    calc_extra_ptm=settings["calc_extra_ptm"],
)

# Preserve the full run in Drive for later QA and publication analysis.
drive.mount("/content/drive")
drive_dir = Path("/content/drive/MyDrive") / BATCH_NAME
if drive_dir.exists():
    from datetime import datetime
    drive_dir = drive_dir.with_name(
        f"{BATCH_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    )
shutil.copytree(RUN_DIR, drive_dir)
archive = shutil.make_archive(str(drive_dir), "zip", root_dir=drive_dir.parent, base_dir=drive_dir.name)
print(f"Completed {len(rows)} candidates")
print(f"Drive folder: {drive_dir}")
print(f"Drive archive: {archive}")


Saving pmhc_colabfold_inputs.fasta to pmhc_colabfold_inputs.fasta
ColabFold is missing in this runtime. Installing it now...


2026-08-01 22:19:10,912 Running on GPU
2026-08-01 22:19:11,265 Found 5 citations for tools or databases
2026-08-01 22:19:11,265 Query 1/86: EBV_TCELL_1393562 (length 377)


COMPLETE: 100%|██████████| 450/450 [elapsed: 00:01 remaining: 00:00]
/usr/local/lib/python3.12/dist-packages/alphafold/model/modules_multimer.py:114: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'> requested in broadcasted_iota is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  iota = jax.lax.broadcasted_iota(jnp.int64, logits.shape, axis)


2026-08-01 22:20:36,302 alphafold2_multimer_v3_model_1_seed_000 recycle=0 pLDDT=90.2 pTM=0.822 ipTM=0.88
2026-08-01 22:21:43,140 alphafold2_multimer_v3_model_1_seed_000 recycle=1 pLDDT=92.6 pTM=0.915 ipTM=0.899 tol=1.45
2026-08-01 22:21:47,325 alphafold2_multimer_v3_model_1_seed_000 recycle=2 pLDDT=93 pTM=0.915 ipTM=0.903 tol=0.538
2026-08-01 22:21:51,502 alphafold2_multimer_v3_model_1_seed_000 recycle=3 pLDDT=93.1 pTM=0.915 ipTM=0.902 tol=1.16
2026-08-01 22:21:56,015 alphafold2_multimer_v3_model_1_seed_000 took 147.1s (3 recycles)
2026-08-01 22:22:00,254 alphafold2_multimer_v3_model_2_seed_000 recycle=0 pLDDT=90 pTM=0.861 ipTM=0.872
2026-08-01 22:22:04,448 alphafold2_multimer_v3_model_2_seed_000 recycle=1 pLDDT=93.2 pTM=0.914 ipTM=0.905 tol=0.806
2026-08-01 22:22:08,626 alphafold2_multimer_v3_model_2_seed_000 recycle=2 pLDDT=95.8 pTM=0.923 ipTM=0.922 tol=0.641
2026-08-01 22:22:12,780 alphafold2_multimer_v3_model_2_seed_000 recycle=3 pLDDT=94.1 pTM=0.913 ipTM=0.903 tol=0.148
2026-08-01

PENDING:   0%|          | 0/450 [elapsed: 00:00 remaining: ?]

...

COMPLETE: 100%|██████████| 450/450 [elapsed: 01:00 remaining: 00:00]
/usr/local/lib/python3.12/dist-packages/alphafold/model/modules_multimer.py:114: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'> requested in broadcasted_iota is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  iota = jax.lax.broadcasted_iota(jnp.int64, logits.shape, axis)
/usr/local/lib/python3.12/dist-packages/alphafold/model/modules_multimer.py:114: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'> requested in broadcasted_iota is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  iota = jax.lax.broadcasted_iota(jnp.int64, l

2026-08-02 04:47:35,633 alphafold2_multimer_v3_model_1_seed_000 recycle=0 pLDDT=83.4 pTM=0.798 ipTM=0.782


/usr/local/lib/python3.12/dist-packages/alphafold/model/modules_multimer.py:114: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'> requested in broadcasted_iota is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  iota = jax.lax.broadcasted_iota(jnp.int64, logits.shape, axis)
/usr/local/lib/python3.12/dist-packages/alphafold/model/modules_multimer.py:114: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'> requested in broadcasted_iota is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  iota = jax.lax.broadcasted_iota(jnp.int64, logits.shape, axis)


2026-08-02 04:48:41,504 alphafold2_multimer_v3_model_1_seed_000 recycle=1 pLDDT=89.5 pTM=0.884 ipTM=0.84 tol=4.77
2026-08-02 04:48:46,193 alphafold2_multimer_v3_model_1_seed_000 recycle=2 pLDDT=90.2 pTM=0.892 ipTM=0.863 tol=3.2
2026-08-02 04:48:50,885 alphafold2_multimer_v3_model_1_seed_000 recycle=3 pLDDT=89.6 pTM=0.891 ipTM=0.861 tol=1.04
2026-08-02 04:48:53,513 alphafold2_multimer_v3_model_1_seed_000 took 148.1s (3 recycles)
2026-08-02 04:48:58,268 alphafold2_multimer_v3_model_2_seed_000 recycle=0 pLDDT=84.3 pTM=0.828 ipTM=0.793
2026-08-02 04:49:02,972 alphafold2_multimer_v3_model_2_seed_000 recycle=1 pLDDT=89.3 pTM=0.879 ipTM=0.835 tol=2.19
2026-08-02 04:49:07,666 alphafold2_multimer_v3_model_2_seed_000 recycle=2 pLDDT=91 pTM=0.889 ipTM=0.858 tol=1.83
2026-08-02 04:49:12,347 alphafold2_multimer_v3_model_2_seed_000 recycle=3 pLDDT=89.4 pTM=0.875 ipTM=0.83 tol=0.534
2026-08-02 04:49:12,513 alphafold2_multimer_v3_model_2_seed_000 took 18.9s (3 recycles)
2026-08-02 04:49:17,242 alphafo

ValueError: mount failed

In [8]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

# Instructions <a name="Instructions"></a>
For detailed instructions, tips and tricks, see recently published paper at [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5)

**Quick start**
1. Paste your protein sequence(s) in the input field.
2. Press "Runtime" -> "Run all".
3. The pipeline consists of 5 steps. The currently running step is indicated by a circle with a stop sign next to it.

**Result zip file contents**

1. PDB formatted structures sorted by avg. pLDDT and complexes are sorted by pTMscore. (unrelaxed and relaxed if `use_amber` is enabled).
2. Plots of the model quality.
3. Plots of the MSA coverage.
4. Parameter log file.
5. A3M formatted input MSA.
6. A `predicted_aligned_error_v1.json` using [AlphaFold-DB's format](https://alphafold.ebi.ac.uk/faq#faq-7) and a `scores.json` for each model which contains an array (list of lists) for PAE, a list with the average pLDDT and the pTMscore.
7. BibTeX file with citations for all used tools and databases.

At the end of the job a download modal box will pop up with a `jobname.result.zip` file. Additionally, if the `save_to_google_drive` option was selected, the `jobname.result.zip` will be uploaded to your Google Drive.

**MSA generation for complexes**

For the complex prediction we use unpaired and paired MSAs. Unpaired MSA is generated the same way as for the protein structures prediction by searching the UniRef100 and environmental sequences three iterations each.

The paired MSA is generated by searching the UniRef100 database and pairing the best hits sharing the same NCBI taxonomic identifier (=species or sub-species). We only pair sequences if all of the query sequences are present for the respective taxonomic identifier.

**Using a custom MSA as input**

To predict the structure with a custom MSA (A3M formatted): (1) Change the `msa_mode`: to "custom", (2) Wait for an upload box to appear at the end of the "MSA options ..." box. Upload your A3M. The first fasta entry of the A3M must be the query sequence without gaps.

It is also possilbe to provide custom MSAs for complex predictions. Read more about the format [here](https://github.com/sokrypton/ColabFold/issues/76).

As an alternative for MSA generation the [HHblits Toolkit server](https://toolkit.tuebingen.mpg.de/tools/hhblits) can be used. After submitting your query, click "Query Template MSA" -> "Download Full A3M". Download the A3M file and upload it in this notebook.

**PDB100** <a name="pdb100"></a>

As of 23/06/08, we have transitioned from using the PDB70 to a 100% clustered PDB, the PDB100. The construction methodology of PDB100 differs from that of PDB70.

The PDB70 was constructed by running each PDB70 representative sequence through [HHblits](https://github.com/soedinglab/hh-suite) against the [Uniclust30](https://uniclust.mmseqs.com/). On the other hand, the PDB100 is built by searching each PDB100 representative structure with [Foldseek](https://github.com/steineggerlab/foldseek) against the [AlphaFold Database](https://alphafold.ebi.ac.uk).

To maintain compatibility with older Notebook versions and local installations, the generated files and API responses will continue to be named "PDB70", even though we're now using the PDB100.

**Using custom templates** <a name="custom_templates"></a>

To predict the structure with a custom template (PDB or mmCIF formatted): (1) change the `template_mode` to "custom" in the execute cell and (2) wait for an upload box to appear at the end of the "Input Protein" box. Select and upload your templates (multiple choices are possible).

* Templates must follow the four letter PDB naming with lower case letters.

* Templates in mmCIF format must contain `_entity_poly_seq`. An error is thrown if this field is not present. The field `_pdbx_audit_revision_history.revision_date` is automatically generated if it is not present.

* Templates in PDB format are automatically converted to the mmCIF format. `_entity_poly_seq` and `_pdbx_audit_revision_history.revision_date` are automatically generated.

If you encounter problems, please report them to this [issue](https://github.com/sokrypton/ColabFold/issues/177).

**Comparison to the full AlphaFold2 and AlphaFold2 Colab**

This notebook replaces the homology detection and MSA pairing of AlphaFold2 with MMseqs2. For a comparison against the [AlphaFold2 Colab](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb) and the full [AlphaFold2](https://github.com/deepmind/alphafold) system read our [paper](https://www.nature.com/articles/s41592-022-01488-1).

**Troubleshooting**
* Check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check your input sequence.

**Known issues**
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Your browser can block the pop-up for downloading the result file. You can choose the `save_to_google_drive` option to upload to Google Drive instead or manually download the result file: Click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

**Limitations**
* Computing resources: Our MMseqs2 API can handle ~20-50k requests per day.
* MSAs: MMseqs2 is very precise and sensitive but might find less hits compared to HHblits/HMMer searched against BFD or MGnify.
* We recommend to additionally use the full [AlphaFold2 pipeline](https://github.com/deepmind/alphafold).

**Description of the plots**
*   **Number of sequences per position** - We want to see at least 30 sequences per position, for best performance, ideally 100 sequences.
*   **Predicted lDDT per position** - model confidence (out of 100) at each position. The higher the better.
*   **Predicted Alignment Error** - For homooligomers, this could be a useful metric to assess how confident the model is about the interface. The lower the better.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/sokrypton/ColabFold/issues

**License**

The source code of ColabFold is licensed under [MIT](https://raw.githubusercontent.com/sokrypton/ColabFold/main/LICENSE). Additionally, this notebook uses the AlphaFold2 source code and its parameters licensed under [Apache 2.0](https://raw.githubusercontent.com/deepmind/alphafold/main/LICENSE) and [CC BY 4.0](https://creativecommons.org/licenses/by-sa/4.0/) respectively. Read more about the AlphaFold license [here](https://github.com/deepmind/alphafold).

**Acknowledgments**
- We thank the AlphaFold team for developing an excellent model and open sourcing the software.

- [KOBIC](https://kobic.re.kr) and [Söding Lab](https://www.mpinat.mpg.de/soeding) for providing the computational resources for the MMseqs2 MSA server.

- Richard Evans for helping to benchmark the ColabFold's Alphafold-multimer support.

- [David Koes](https://github.com/dkoes) for his awesome [py3Dmol](https://3dmol.csb.pitt.edu/) plugin, without whom these notebooks would be quite boring!

- Do-Yoon Kim for creating the ColabFold logo.

- A colab by Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)) and Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger)).
